# Wind tutorial

In [ ]:
import enn554.wind as wind
import matplotlib.pyplot as plt
from enn554.paths import data_dir
import numpy as np
import pandas as pd
dd = data_dir()

In [ ]:
coopers_gap = wind.merra_wind_speed_data()
coopers_gap.import_data(dd / 'tutorial_4/POWER_Point_Hourly_20150101_20251231_026d73S_151d47E_UTC.csv')
coopers_gap.utc_to_lst(10)
coopers_gap.data

In [ ]:
ax = wind.wind_rose(coopers_gap.data['WD50M'], coopers_gap.data['WS50M'],16)
ax.set_title('Coopers Gap Wind Rose at 50m')

Change the height and do the wind rose again!

In [ ]:
coopers_gap.add_speed_at_height(100,alpha=1/7.0)

In [ ]:
ax = wind.wind_rose(coopers_gap.data['WD100M'], coopers_gap.data['WS100M'],16)
ax.set_title('Coopers Gap Wind Rose at 100m')

mean_ws_50m = coopers_gap.data['WS50M'].mean()
mean_ws_100m = coopers_gap.data['WS100M'].mean()
print(f'Mean wind speed at 50m: {mean_ws_50m:.2f} m/s')
print(f'Mean wind speed at 100m: {mean_ws_100m:.2f} m/s')

# coopers_gap.data['WD50M'].mean() WRONG!!!!!!!
mean_direction_50m = wind.mean_direction(coopers_gap.data['WD50M'])
mean_direction_100m = wind.mean_direction(coopers_gap.data['WD100M'])
print(f'Mean wind direction at 50m: {mean_direction_50m:.2f} degrees')
print(f'Mean wind direction at 100m: {mean_direction_100m:.2f} degrees')


In [ ]:
dist = wind.speed_fit(coopers_gap.data['WS100M'],plot=False,type='weibull',
                      hist_kwargs={'bins':20})

fig,ax = plt.subplots()
ax.hist(coopers_gap.data['WS100M'], bins=20, density=True, alpha=0.6, color='g')
x = np.linspace(0, coopers_gap.data['WS100M'].max(), 100)
pdf = dist.pdf(x)
ax.plot(x, pdf, 'r-', lw=2, label='Weibull PDF')
ax.set_xlabel('Wind Speed (m/s)')
ax.set_title(f'Wind Speed Distribution at 100m ({dist.mean():.2f} m/s)')



In [ ]:
model = wind.speed_and_direction_dist(type='weibull',n_az_bins=16)
model.fit(coopers_gap.data['WD100M'], coopers_gap.data['WS100M'],plot=False)
model.print_params()

Wind power density

In [ ]:
T_sea_level = 15+273.15
P_sea_level = 101.3
P = P_sea_level - (0.011837) * 100 + (4.793e-7) * 100**2
T_100m = T_sea_level - 0.66
rho_air = (P*1000) / (287.05 * T_100m)

In [ ]:
WPD=0.5 * rho_air * (mean_ws_100m**3)
print(f'Wind Power Density at 100m: {WPD:.2f} W/m^2')

In [ ]:
sorted_ws = np.sort(coopers_gap.data['WS100M'].values)
sorted_ws = np.flip(sorted_ws)
hours_above = np.arange(1, len(sorted_ws) + 1)
fig, ax = plt.subplots()

hours_one_year = hours_above * 8760/len(hours_above)
ax.plot(hours_one_year, sorted_ws)
ax.set_xlabel('Hours per Year')
ax.set_ylabel('Wind Speed (m/s)')
ax.set_title('Wind Speed Duration Curve at 100m')